# lyricsLM: build a GPT from scratch

Companion notebook to the article. Builds and trains a small GPT-style transformer end to end: data, tokenizer, attention, training loop, generation, and a from-scratch NumPy reimplementation of the forward pass.

**Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**. Takes about 10 minutes.

This is the standalone version of the notebook: it trains the model and exports the weights, and stops there. It doesn't include the Google Drive, GitHub, or deployment steps used to publish the live demo — those are specific to how the author hosts the site.

In [ ]:
import torch, torch.nn as nn, requests, re, time
from torch.nn import functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
if device == 'cpu':
    print('WARNING: no GPU. Go to Runtime > Change runtime type > T4 GPU.')
    print('On CPU this will take hours instead of minutes.')
import matplotlib.pyplot as plt
import json, os, shutil

## 1. Get the songbooks

In [ ]:
BOOKS = {
    # ---- Indian poets, about 30% of the corpus ----
    'tagore_gitanjali':     7164,   # song offerings, the Nobel book
    'tagore_gardener':      6686,   # love poems
    'tagore_stray_birds':   6524,   # short aphorisms
    'tagore_fruit':         6522,   # Fruit-Gathering
    'tagore_crescent':      6520,   # The Crescent Moon
    'tagore_fugitive':      7971,   # The Fugitive
    'kabir_songs':          6519,   # Songs of Kabir, Tagore's translation
    'indian_anthology':    74751,   # Anthology of Modern Indian Poetry
    'naidu_threshold':       680,   # Sarojini Naidu

    # ---- Western poets, about 70% ----
    'dickinson':           12242,   # Poems, Three Series, Complete
    'poe':                 10031,   # Complete Poetical Works
    'frost_new_hampshire': 58611,   # 1923, contains our seed line
    'blake_songs':          1934,   # Songs of Innocence and of Experience
    'keats_1820':          23684,   # Poems Published in 1820
    'wordsworth':           8774,   # Poems in Two Volumes, Volume 1
    'wordsworth_tennyson': 14952,   # Selections
}
# Frost is public domain in the US. Outside the US he is in copyright
# until 2034, so delete that line if you are elsewhere.
# Do NOT add Shelley #4800 without reading the balance section: it is
# larger than everything above combined.

URL_PATTERNS = [
    'https://www.gutenberg.org/ebooks/{id}.txt.utf-8',
    'https://www.gutenberg.org/files/{id}/{id}-0.txt',
    'https://www.gutenberg.org/files/{id}/{id}.txt',
    'https://www.gutenberg.org/cache/epub/{id}/pg{id}.txt',
]
HEADERS = {'User-Agent': 'Mozilla/5.0 (compatible; lyricsLM-tutorial/1.0)'}

def fetch(book_id):
    for pattern in URL_PATTERNS:
        url = pattern.format(id=book_id)
        try:
            r = requests.get(url, headers=HEADERS, timeout=30)
            if r.status_code == 200 and len(r.text) > 5000:
                return r.text
        except Exception:
            pass
        time.sleep(1)
    raise RuntimeError(f'could not download book {book_id}')

raw = {}
for name, bid in BOOKS.items():
    raw[name] = fetch(bid)
    print(f'{name:24s} {len(raw[name]):>9,} chars')

## 2. Clean it

In [ ]:
def clean(text):
    # tail first, or the head cut shifts every offset after it
    end = re.search(r'\*\*\*\s*END OF (THE|THIS) PROJECT GUTENBERG.*?\*\*\*', text, re.S | re.I)
    if end:   text = text[:end.start()]
    start = re.search(r'\*\*\*\s*START OF (THE|THIS) PROJECT GUTENBERG.*?\*\*\*', text, re.S | re.I)
    if start: text = text[start.end():]

    text = text.replace('\r\n', '\n')
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    return text.strip()

cleaned = {n: clean(t) for n, t in raw.items()}
corpus = '\n\n'.join(cleaned.values())
print(f'{len(corpus):,} characters')
print()
print('--- your real numbers ---')
print('characters per step :', 128*64)
print('steps for one pass  :', round(len(corpus)/(128*64)))
print('20000 steps         :', round(20000/(len(corpus)/(128*64))), 'passes')
print()
print(corpus[:400])

print()
print('BALANCE  (no single book should be far above 25%)')
for name, txt in sorted(cleaned.items(), key=lambda kv: -len(kv[1])):
    share = len(txt)/len(corpus)*100
    flag = '   <-- too big' if share > 25 else ''
    print(f'  {name:24s} {share:5.1f}%{flag}')
indian = sum(len(t) for n,t in cleaned.items()
             if n.startswith(('tagore','kabir','indian','naidu')))
print(f'  Indian poets combined  {indian/len(corpus)*100:.0f}%')

## 3. Tokenize

In [ ]:
chars = sorted(set(corpus))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda nums: ''.join(itos[i] for i in nums)

print('vocab size:', vocab_size)
print(encode('miles to go'))
print(decode(encode('miles to go')))

## 4. Batches

In [ ]:
block_size = 128
batch_size = 64
n_embd = 192
n_head = 6
n_layer = 6
dropout = 0.2
learning_rate = 3e-4
max_steps = 20000

data = torch.tensor(encode(corpus), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size - 1, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch('train')
print(xb.shape, yb.shape)

## 5. Attention

In [ ]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k, q = self.key(x), self.query(x)
        w = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
        w = w.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        w = F.softmax(w, dim=-1)
        w = self.dropout(w)
        return w @ self.value(x)

class MultiHead(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

## 6. Block and model

The architecture here (Head, MultiHead, Block, the general shape of a small GPT) follows the pattern from Andrej Karpathy's "Let's Build GPT" walkthrough and nanoGPT. I adapted it for this corpus and vocabulary rather than designing the block structure from a blank page, it's a well trodden path for a reason.


In [ ]:
class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd), nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd), nn.Dropout(dropout))
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.sa = MultiHead(n_head, n_embd // n_head)
        self.ff = FeedForward()
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)  # separate norm before attention vs before ffwd, pre-norm style
    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

class LyricsLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block() for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok = self.token_embedding(idx)
        pos = self.position_embedding(torch.arange(T, device=idx.device))
        x = self.blocks(tok + pos)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(B*T, -1), targets.view(B*T))
        return logits, loss

model = LyricsLM().to(device)
print(f'{sum(p.numel() for p in model.parameters()):,} parameters')

## 7. Train

In [ ]:
@torch.no_grad()
def estimate_loss(iters=50):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(iters)
        for k in range(iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
history = []

torch.save(model.state_dict(), 'checkpoint_0.pt')

for step in range(max_steps):
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % 500 == 0:
        l = estimate_loss()
        history.append((step, l['train'], l['val']))
        print(f"step {step:5d} | train {l['train']:.4f} | val {l['val']:.4f}")

    if step == max_steps // 2:
        torch.save(model.state_dict(), f'checkpoint_{max_steps // 2}.pt')

torch.save(model.state_dict(), 'checkpoint_final.pt')
print('done')

## 8. The loss curve

In [ ]:

steps = [h[0] for h in history]
plt.figure(figsize=(8,4))
plt.plot(steps, [h[1] for h in history], label='train', color='#1D9E75')
plt.plot(steps, [h[2] for h in history], label='validation', color='#0F6E56', linestyle='--')
plt.xlabel('training step'); plt.ylabel('cross-entropy loss')
plt.title('the report card'); plt.legend(); plt.grid(alpha=0.3)
plt.show()

## 9. Generate

In [ ]:
@torch.no_grad()
def generate(model, seed, max_new_tokens=300, temperature=0.8):
    model.eval()
    idx = torch.tensor([encode(seed)], dtype=torch.long, device=device)
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -block_size:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / temperature
        probs = F.softmax(logits, dim=-1)
        nxt = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, nxt), dim=1)
    return decode(idx[0].tolist())

print(generate(model, 'and miles to go before I sleep'))

## 10. Three checkpoints, same seed line

In [ ]:
seed = 'and miles to go before I sleep'   # <-- put your own line here

half_step = max_steps // 2
for name, path in [('baby', 'checkpoint_0.pt'),
                   ('halfway', f'checkpoint_{half_step}.pt'),
                   ('fully trained', 'checkpoint_final.pt')]:
    model.load_state_dict(torch.load(path))
    print(f'\n===== {name} =====')
    print(generate(model, seed, max_new_tokens=200))

## 11. Play with the creativity dial

In [ ]:
# Make sure we're generating from the fully trained model
model.load_state_dict(torch.load('checkpoint_final.pt'))

for t in [0.4, 0.8, 1.2]:
    print(f'\n--- temperature {t} ---')
    print(generate(model, 'the night is deep', max_new_tokens=150, temperature=t))

## 12. Same forward pass, no PyTorch

nn.Linear, nn.Embedding, nn.LayerNorm, F.softmax are just matrix ops underneath. Training needs PyTorch for autodiff (nobody's hand-computing gradients through six transformer blocks), but generation doesn't need any of that machinery. It's multiply, add, softmax, sample.

Below I rebuild the section 6 forward pass in plain NumPy, same weights, same architecture. If the numbers match PyTorch, it's confirmation nothing in nn.Module is hiding extra logic.


In [ ]:
import numpy as np

# Pull the trained weights out of PyTorch's state_dict into plain NumPy arrays.
# From here on, nothing below touches torch at all.
W = {k: v.detach().cpu().numpy() for k, v in model.state_dict().items()}

def np_layer_norm(x, weight, bias, eps=1e-5):
    mean = x.mean(axis=-1, keepdims=True)
    var = x.var(axis=-1, keepdims=True)
    return (x - mean) / np.sqrt(var + eps) * weight + bias

def np_linear(x, weight, bias=None):
    # nn.Linear stores weight as (out_features, in_features), hence the transpose
    out = x @ weight.T
    return out + bias if bias is not None else out

def np_softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)  # numerical stability
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

In [ ]:
def np_attention_head(x, b, h, T):
    p = f'blocks.{b}.sa.heads.{h}'
    k = np_linear(x, W[f'{p}.key.weight'])
    q = np_linear(x, W[f'{p}.query.weight'])
    v = np_linear(x, W[f'{p}.value.weight'])
    wei = q @ k.transpose(0, 2, 1) * (k.shape[-1] ** -0.5)
    mask = np.tril(np.ones((T, T)))
    wei = np.where(mask == 0, -1e10, wei)  # same masking idea as the torch version
    wei = np_softmax(wei, axis=-1)
    return wei @ v

def np_multi_head(x, b, T):
    outs = [np_attention_head(x, b, h, T) for h in range(n_head)]
    out = np.concatenate(outs, axis=-1)
    return np_linear(out, W[f'blocks.{b}.sa.proj.weight'], W[f'blocks.{b}.sa.proj.bias'])

def np_feed_forward(x, b):
    p = f'blocks.{b}.ff.net'
    x = np_linear(x, W[f'{p}.0.weight'], W[f'{p}.0.bias'])
    x = np.maximum(x, 0)  # ReLU
    return np_linear(x, W[f'{p}.2.weight'], W[f'{p}.2.bias'])

def np_block(x, b, T):
    ln1w = W[f'blocks.{b}.ln1.weight']
    ln1b = W[f'blocks.{b}.ln1.bias']
    ln2w = W[f'blocks.{b}.ln2.weight']
    ln2b = W[f'blocks.{b}.ln2.bias']
    x = x + np_multi_head(np_layer_norm(x, ln1w, ln1b), b, T)
    x = x + np_feed_forward(np_layer_norm(x, ln2w, ln2b), b)
    return x

def numpy_forward(idx):
    B, T = idx.shape
    tok = W['token_embedding.weight'][idx]
    pos = W['position_embedding.weight'][:T]
    x = tok + pos
    for i in range(n_layer):
        x = np_block(x, i, T)
    x = np_layer_norm(x, W['ln_f.weight'], W['ln_f.bias'])
    return np_linear(x, W['head.weight'], W['head.bias'])

### Does it actually match

Same input through both versions, compare the logits. They should agree to floating point precision if the NumPy version is a faithful translation.


In [ ]:
idx_check = torch.tensor([encode('and miles to go')], dtype=torch.long).to(device)

with torch.no_grad():
    torch_logits, _ = model(idx_check)

np_logits = numpy_forward(idx_check.cpu().numpy())

diff = np.max(np.abs(torch_logits.cpu().numpy() - np_logits))
print(f'max abs difference: {diff:.2e}')
print('MATCH — the NumPy version is the same function, just without a framework.' if diff < 1e-3 else 'MISMATCH — something above doesn\'t line up.')

### Generating from the NumPy version

Same loop as generate() in section 9, just calling numpy_forward instead of model(...).


In [ ]:
def generate_numpy(seed, max_new_tokens=200, temperature=0.8):
    idx = np.array([encode(seed)], dtype=np.int64)
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -block_size:]
        logits = numpy_forward(idx_cond)
        logits = logits[:, -1, :] / temperature
        probs = np_softmax(logits, axis=-1)[0]
        nxt = np.random.choice(len(probs), p=probs)
        idx = np.concatenate([idx, np.array([[nxt]])], axis=1)
    return decode(idx[0].tolist())

print(generate_numpy('and miles to go before I sleep'))

## 13. Exporting the weights

The live demo runs on a small server with no PyTorch installed — inference doesn't need it. This cell dumps the trained weights to a plain `.npz` file plus a `config.json` with the vocab and architecture settings needed to rebuild the model, saved locally to an `export/` folder.

In [ ]:
# Make sure we're exporting the fully trained model
model.load_state_dict(torch.load('checkpoint_final.pt'))

npz_dict = {k: v.detach().cpu().numpy().astype(np.float32) for k, v in model.state_dict().items()}

MODEL_DIR = 'export'
os.makedirs(MODEL_DIR, exist_ok=True)

npz_path = os.path.join(MODEL_DIR, 'trained_weights.npz')
np.savez(npz_path, **npz_dict)
print(f'Saved {len(npz_dict)} tensors -> {npz_path}  ({os.path.getsize(npz_path)/1e6:.1f} MB)')

config = {
    'vocab_size': vocab_size,
    'n_embd': n_embd,
    'n_head': n_head,
    'n_layer': n_layer,
    'block_size': block_size,
    'stoi': stoi,
    'itos': {str(k): v for k, v in itos.items()},
}
config_path = os.path.join(MODEL_DIR, 'config.json')
with open(config_path, 'w') as f:
    json.dump(config, f)
print('config.json written ->', config_path)